In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("data.csv")

In [3]:
df.head()

,data
0,Please check the status of your password reset...
1,Our support team can help you resolve the pass...
2,"If the password reset fails again, please cont..."
3,We are reviewing your password reset and will ...
4,Please provide more details about the password...


In [4]:
df.shape

(1200, 1)

In [5]:
data = df['data']
data.head()

0    Please check the status of your password reset...
1    Our support team can help you resolve the pass...
2    If the password reset fails again, please cont...
3    We are reviewing your password reset and will ...
4    Please provide more details about the password...
Name: data, dtype: object

In [6]:
data = data.str.lower()

In [8]:
import string
trans = str.maketrans('', '', string.punctuation)
data = data.apply(lambda x: x.translate(trans))

In [9]:
data.head()

0    please check the status of your password reset...
1    our support team can help you resolve the pass...
2    if the password reset fails again please conta...
3    we are reviewing your password reset and will ...
4    please provide more details about the password...
Name: data, dtype: object

In [12]:
!pip install tensorflow

In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [74]:
vocab_size = 208

tokinizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokinizer.fit_on_texts(data)

vocab_size = len(tokinizer.word_index) + 1
print("Vocabulary size:", vocab_size)

Vocabulary size: 209


In [75]:
word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10]

208


[('<OOV>', 1),
 ('the', 2),
 ('your', 3),
 ('please', 4),
 ('you', 5),
 ('support', 6),
 ('we', 7),
 ('to', 8),
 ('is', 9),
 ('before', 10)]

In [76]:
sequence = tokinizer.texts_to_sequences(data)

In [77]:
for i in range(3):
  print(data[i])

please check the status of your password reset before trying again
our support team can help you resolve the password reset
if the password reset fails again please contact technical support


In [78]:
for i in range(3):
  print(sequence[i])

[4, 40, 2, 35, 28, 3, 36, 67, 10, 108, 14]
[11, 6, 15, 20, 29, 5, 41, 2, 36, 67]
[30, 2, 36, 67, 109, 14, 4, 42, 43, 6]


In [80]:
max_len=max(len(x) for x in X)
print(max_len)

12


In [115]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

X = []
y = []

for seq in sequence:
    for i in range(1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]

        X.append(input_seq)
        y.append(output_seq)

X = pad_sequences(X, maxlen=max_len, padding="pre")
X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (12120, 12)
y shape: (12120,)


In [116]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_pad = pad_sequences(X, maxlen=max_len, padding='pre')


In [117]:
y = np.array(y)

In [118]:
X_pad.shape

(12120, 12)

In [119]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [120]:
y.shape

(12120,)

In [121]:
y_one_hot.shape

(12120, 209)

In [122]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [123]:
embed_dim = 40
rnn_units = 116

In [124]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

In [126]:
rnn_model = Sequential()

rnn_model.add(Input(shape=(max_len,)))
rnn_model.add(Embedding(input_dim=vocab_size, output_dim=embed_dim))
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

In [127]:
rnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [128]:
rnn_model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)              │ (None, 12, 40)              │           8,360 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_4 (SimpleRNN)             │ (None, 116)                 │          18,212 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 209)                 │          24,453 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 51,025 (199.32 KB)

 Trainable params: 51,025 (199.32 KB)

 Non-trainable params: 0 (0.00 B)

In [129]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

lstm_model = Sequential()

lstm_model.add(Input(shape=(max_len,)))
lstm_model.add(Embedding(input_dim=vocab_size, output_dim=embed_dim))
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [130]:
lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [139]:
history = lstm_model.fit(
    X,
    y,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
303/303 ━━━━━━━━━━━━━━━━━━━━ 13s 33ms/step - accuracy: 0.0884 - loss: 4.7657 - val_accuracy: 0.1394 - val_loss: 4.7832
Epoch 2/100
303/303 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.2632 - loss: 3.2883 - val_accuracy: 0.4163 - val_loss: 3.4896
Epoch 3/100
303/303 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.5850 - loss: 1.8963 - val_accuracy: 0.6786 - val_loss: 2.8170
Epoch 4/100
303/303 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.7530 - loss: 1.2023 - val_accuracy: 0.7186 - val_loss: 2.5815
Epoch 5/100
303/303 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.8095 - loss: 0.8754 - val_accuracy: 0.7162 - val_loss: 2.5253
Epoch 6/100
303/303 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - accuracy: 0.8285 - loss: 0.7044 - val_accuracy: 0.7215 - val_loss: 2.5259
Epoch 7/100
303/303 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.8293 - loss: 0.6180 - val_accuracy: 0.7182 - val_loss: 2.5543
Epoch 8/100
303/303 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.8309 - loss: 0.5751 

In [140]:
lstm_model.summary()

Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)              │ (None, 12, 40)              │           8,360 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ (None, 116)                 │          72,848 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 209)                 │          24,453 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 316,985 (1.21 MB)

 Trainable params: 105,661 (412.74 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 211,324 (825.49 KB)

In [141]:
lstm_model.save("lstm.keras")

In [142]:
print("Model output size:", lstm_model.output_shape[-1])
print("Tokenizer vocabulary size:", len(tokinizer.word_index))
print("Index 1566 exists:", 1566 in index_to_word)

Model output size: 209
Tokenizer vocabulary size: 208
Index 1566 exists: False


In [143]:
index_to_word = {
    index: word
    for word, index in tokinizer.word_index.items()
}

print("Model output size:", lstm_model.output_shape[-1])
print("Tokenizer vocabulary size:", len(tokinizer.word_index))
print("Index 1566 exists:", 1566 in index_to_word)

Model output size: 209
Tokenizer vocabulary size: 208
Index 1566 exists: False


In [144]:
from tensorflow.keras.models import load_model

lstm_model = load_model("lstm.keras")
print(lstm_model.output_shape)

(None, 209)


In [145]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [146]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predictor(model, tokenizer, text, max_len):
    sequence = tokenizer.texts_to_sequences([text])[0]

    if not sequence:
        return ""

    seq = pad_sequences(
        [sequence],
        maxlen=max_len,
        padding="pre",
        truncating="pre"
    )

    pred = model.predict(seq, verbose=0)[0]

    # Ignore padding and unknown indices
    pred[:1] = 0
    pred_index = int(np.argmax(pred))

    index_to_word = {
        index: word for word, index in tokenizer.word_index.items()
    }

    return index_to_word.get(pred_index, "")

In [147]:
seed = "The password reset may"

generated_text = generate_text(
    lstm_model,
    tokinizer,
    seed,
    max_len,
    5
)

print(generated_text)

The password reset may take a few minutes to


In [151]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokinizer, f)

In [152]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)

In [153]:
import pickle
lstm_model.save("lstm.keras")
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokinizer, f)
print("Model and tokenizer saved successfully!")

Model and tokenizer saved successfully!
